# Donut – chạy dự án trên Colab / Kaggle
Chạy từng cell theo thứ tự. Xem chi tiết trong `README.md`. Checkpoint train luôn ghi lên Google Drive (`DRIVE_OUT`).

## 0. Setup

In [ ]:
# Colab: mount Drive và vào thư mục dự án (sửa đường dẫn nếu khác)
from google.colab import drive
drive.mount('/content/drive')
PROJECT = '/content/drive/MyDrive/donut'
DRIVE_OUT = '/content/drive/MyDrive/donut/outputs'
%cd {PROJECT}

In [ ]:
# Kaggle (thay cho cell trên):
# !cp -r /kaggle/input/<ten-dataset-du-an> /kaggle/working/donut
# %cd /kaggle/working/donut
# DRIVE_OUT = '/kaggle/working/outputs'

In [ ]:
!nvidia-smi
!bash scripts/phase0_setup.sh

## Phase 1 – Tái lặp
### 1.1 Checkpoint chính thức trên CORD-v2 test (~15 phút)

In [ ]:
!bash scripts/phase1_reproduce.sh

In [ ]:
import pandas as pd
pd.read_csv('results/p1_summary.csv')

### 1.2 Smoke test pipeline train (~3–5 phút)

In [ ]:
!python -m src.train --config configs/smoke_test.yaml

### 1.3 Fine-tune B0 (3–5 giờ). Bị ngắt → chạy lại cell này để resume.

In [ ]:
!bash scripts/phase1_finetune.sh {DRIVE_OUT}/B0_full_1280

In [ ]:
from IPython.display import Image, display
display(Image('results/figures/B0_curves.png'))

### 1.4 (tùy chọn) RVL-CDIP / DocVQA trên tập con

In [ ]:
# !python -m src.eval_multitask --task rvlcdip --limit 500
# !python -m src.eval_multitask --task docvqa --limit 500

## Phase 2 – How does Donut work?

In [ ]:
!bash scripts/phase2_analysis.sh

In [ ]:
import glob
display(Image('results/figures/architecture.png'))
for f in sorted(glob.glob('results/attention/*_summary.png')):
    print(f); display(Image(f))
print(open('results/error_analysis.md', encoding='utf-8').read())

## Phase 3 – Cải tiến
### I1 – độ phân giải × tốc độ

In [ ]:
!bash scripts/phase3_i1_resolution.sh
display(Image('results/figures/i1_tradeoff.png'))

### I4 – beam search / hậu xử lý / giải mã ràng buộc

In [ ]:
!bash scripts/phase3_i4_decoding.sh

### I2 – LoRA (2–3 giờ)

In [ ]:
!bash scripts/phase3_i2_lora.sh {DRIVE_OUT}/I2_lora_r8

### I3 – augmentation + độ bền (cần B0 xong)

In [ ]:
!bash scripts/phase3_i3_robustness.sh {DRIVE_OUT}/B0_full_1280/best {DRIVE_OUT}

### I5 – fp16 / int8 (tùy chọn)

In [ ]:
# !bash scripts/phase3_i5_compress.sh naver-clova-ix/donut-base-finetuned-cord-v2 30

## Tổng hợp bảng kết quả

In [ ]:
!bash scripts/make_tables.sh
pd.read_csv('results/runs.csv')